In [1]:
import pandas as pd 

In [ ]:
# 1. Ruta
ruta = "../data_raw/libro_mayor_2025.xlsx"

# 2. Cargar datos
df = pd.read_excel(ruta, header=2)

# 3. Limpieza de espacios en las columnas 
df.columns = df.columns.str.strip()

# 4. Tipos de datos de las columnas

# a. de la columna Fecha de factura cambiar el formato de fecha y decirle que la lea en formato día, mes y año.
df ["Fecha de factura"] = pd.to_datetime(df ["Fecha de factura"], 
                                         format="%d/%m/%Y",
                                         errors="coerce")

# b. de la columa Valor Final se le transforma el formato a valor numerico.
df ["Valor Final"] = pd.to_numeric(df ["Valor Final"], errors="coerce")

# 5. Rellenar la columna códigos (estructura del dataset)
df['Código'] = df['Código'].ffill()

# 6. convierte los valores a texto, suprime los espacios que contenga la columna y elimina de la columna Nombre de la cuenta el 
# texto que empiece con la palabra Total
df = df[
    ~df["Nombre de la cuenta"]
    .astype(str)
    .str.strip()
    .str.startswith("Total", na=False)
]
# 7. eliminacion de una fila sin valor representativo, solo informativo
df = df[df["Fecha de factura"].notna() & (df["Fecha de factura"] != "")]

# 8. busca con filtro en la columna Nombre de la cuenta las palabras que coincidan con Balance inicial y agregarle fehca del 
# 31-12-2024 en la columna Fecha de factura 
mask = df["Nombre de la cuenta"].str.contains("Balance inicial", na=False)

df.loc[mask, "Fecha de factura"] = pd.Timestamp("2024-12-31") 

# 9. se crea la columna Contacto_Norm, despues la función convierte cada nombre en un codigo corto y unico para ocultar el nombre real. 
# sino hay nonbre pone sin contacto, luego se guardan estos datos en una columna nueva y se ocultan las otras dos columnas

df["Contacto_Norm"] = df["Contacto"].astype(str).str.upper().str.strip()

import hashlib

def codigo_contacto(nombre):
    if pd.isna(nombre):
        return "SIN_CONTACTO" 
    nombre = str(nombre)
    return "T-" + hashlib.sha256(nombre.encode()).hexdigest()[:5]

df["Contacto_Pseudo"] = df["Contacto_Norm"].apply(codigo_contacto)

df = df.drop(columns=["Contacto", "Contacto_Norm"])

# 10.estandarizar los nombres de las columnas y dejar el dataframe de forma profesional
def estandarizar_columnas(df):
    df.columns = (
        df.columns
        .str.strip()
        .str.lower()
        .str.replace(" ", "_")
        .str.replace("ó", "o")
        .str.replace("é", "e")
        .str.replace("í", "i")
        .str.replace("á", "a")
        .str.replace("ú", "u")
    )
    return df
df = estandarizar_columnas(df)

# 11. Guardar dataset limpio base
df_limpio = df.copy()
df_limpio.to_excel("../data_clean/libro_mayor_2025_limpio.xlsx")

df.head(11)
